# YouTube Morocco Benchmark — ML Pipeline
**Pipeline structure:** Fusion → Cleaning → Feature Engineering → EDA → XGBoost Modeling

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

---
## BLOC 1 — Fusion

In [ ]:
# Load and merge all CSV files from 25 channels
files = glob.glob('data*.csv')
df_list = []

for file in files:
    df_tmp = pd.read_csv(file, encoding='utf-8-sig')
    df_list.append(df_tmp)

df_raw = pd.concat(df_list, ignore_index=True)

print(f'Fusion complete: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print(f'Channels found: {df_raw["channelTitle"].nunique() if "channelTitle" in df_raw.columns else "N/A"}')

In [ ]:
# Keep a clean raw copy — never overwrite it during cleaning
df_raw.to_csv('df_benchmark_raw.csv', index=False)
df = df_raw.copy()
print('Raw snapshot saved to df_benchmark_raw.csv')

In [ ]:
# Post-fusion diagnostic
print('=== Shape ===')
print(df.shape)
print()
print('=== Dtypes ===')
print(df.dtypes)
print()
print('=== Duplicates ===')
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Remove duplicates
df = df.drop_duplicates()
print('Rows after dedup:', df.shape[0])

---
## BLOC 2 — Cleaning

In [ ]:
# Explicit numeric casting before any calculation
numeric_cols = ['viewCount', 'likeCount', 'commentCount', 'durationSec']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Missing value analysis — before any imputation
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)
print('=== Missing Values (before imputation) ===')
print(missing_df)

# Decision rule:
# - > 60% missing: candidate for removal
# - text fields (tags, description): fill with empty string
# - numeric metrics: only fill if truly optional (not key KPIs)
high_missing = missing_df[missing_df['pct'] > 60].index.tolist()
print(f'\nColumns with >60% missing (to review for removal): {high_missing}')

In [ ]:
# Columns to drop — after verifying fill rates above
cols_to_drop = ['dislikeCount', 'locationDescription', 'latitude',
                'longitude', 'hasPaidProductPlacement', 'thumbnail_maxres',
                'favoriteCount']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print('Columns after drop:', df.shape[1])

In [ ]:
# Separate truly zero values from missing for key metrics
# Videos where viewCount is NaN are ambiguous — drop them (< 0.1% of dataset)
n_before = len(df)
df = df.dropna(subset=['viewCount'])
print(f'Dropped {n_before - len(df)} rows with missing viewCount')

# likeCount and commentCount: fill NaN with 0 only after confirming the video exists
# (likes/comments disabled is a real zero, not a data gap)
df['likeCount'] = df['likeCount'].fillna(0)
df['commentCount'] = df['commentCount'].fillna(0)
df['durationSec'] = df['durationSec'].fillna(0)

# Text fields
df['tags'] = df['tags'].fillna('')
df['videoDescription'] = df['videoDescription'].fillna('')
df['topicCategories'] = df['topicCategories'].fillna('Unknown')

# Boolean field
df['licensedContent'] = df['licensedContent'].fillna(0).astype(int)

In [ ]:
# Date parsing
df['publishedAtSQL'] = pd.to_datetime(df['publishedAtSQL'], errors='coerce')
df = df.dropna(subset=['publishedAtSQL'])
df = df.sort_values('publishedAtSQL').reset_index(drop=True)
print('Date range:', df['publishedAtSQL'].min(), '->', df['publishedAtSQL'].max())

In [ ]:
# Outlier analysis on viewCount
print('=== viewCount distribution ===')
print(df['viewCount'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['viewCount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('viewCount — raw')
axes[0].set_xlabel('views')

log_views = np.log1p(df['viewCount'])
axes[1].hist(log_views, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('log1p(viewCount)')
axes[1].set_xlabel('log views')

plt.tight_layout()
plt.show()

---
## BLOC 3 — Feature Engineering

In [ ]:
# Temporal features
df['publish_hour'] = df['publishedAtSQL'].dt.hour
df['publish_day'] = df['publishedAtSQL'].dt.day_name()
df['publish_month'] = df['publishedAtSQL'].dt.month
df['publish_year'] = df['publishedAtSQL'].dt.year
df['is_weekend'] = df['publishedAtSQL'].dt.dayofweek.isin([5, 6]).astype(int)

In [ ]:
# Format features
df['is_short'] = (df['durationSec'] <= 60).astype(int)
df['log_duration'] = np.log1p(df['durationSec'])

In [ ]:
# Engagement features — filter out zero-view videos before computing ratios
zero_view_mask = df['viewCount'] == 0
print(f'Videos with 0 views: {zero_view_mask.sum()} — excluded from ratio calculations')

df['engagement_rate'] = np.where(
    df['viewCount'] > 0,
    (df['likeCount'] + df['commentCount']) / df['viewCount'],
    np.nan
)
df['like_ratio'] = np.where(
    df['viewCount'] > 0,
    df['likeCount'] / df['viewCount'],
    np.nan
)

# Cap outliers at 99th percentile to stabilize the model
er_cap = df['engagement_rate'].quantile(0.99)
df['engagement_rate'] = df['engagement_rate'].clip(upper=er_cap)
print(f'engagement_rate capped at {er_cap:.4f} (99th pct)')

In [ ]:
# Log transform views for model features
df['log_views'] = np.log1p(df['viewCount'])
df['log_likes'] = np.log1p(df['likeCount'])
df['log_comments'] = np.log1p(df['commentCount'])

In [ ]:
# NLP features — title
df['title_length'] = df['videoTitle'].astype(str).apply(lambda x: len(x.split()))
df['title_char_count'] = df['videoTitle'].astype(str).apply(len)

# Emoji detection without external dependency
def has_emoji(text):
    text = str(text)
    for char in text:
        cp = ord(char)
        if (0x1F300 <= cp <= 0x1FAFF) or (0x2600 <= cp <= 0x27BF) or (0xFE00 <= cp <= 0xFE0F):
            return 1
    return 0

df['has_emoji'] = df['videoTitle'].apply(has_emoji)

In [ ]:
# Tags feature — robust parsing for list, string, or NaN formats
def parse_tags_count(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 0
    if isinstance(val, list):
        return len(val)
    if isinstance(val, str):
        val = val.strip()
        if val == '' or val == '[]':
            return 0
        # Handle Python list exported as string: "['tag1', 'tag2']"
        if val.startswith('[') and val.endswith(']'):
            inner = val[1:-1]
            parts = [p.strip().strip("'\"") for p in inner.split(',') if p.strip()]
            return len(parts)
        return len(val.split(','))
    return 0

df['tags_count'] = df['tags'].apply(parse_tags_count)
print('tags_count sample:', df['tags_count'].describe())

In [ ]:
# Target variable — binary classification: is this a high-view video?
view_median = df.loc[df['viewCount'] > 0, 'viewCount'].median()
df['high_view'] = (df['viewCount'] >= view_median).astype(int)

print(f'Median viewCount (non-zero videos): {view_median:,.0f}')
print(f'Target distribution:')
print(df['high_view'].value_counts())
print(df['high_view'].value_counts(normalize=True).round(3))

In [ ]:
# Categorical encoding for XGBoost
le_day = LabelEncoder()
le_channel = LabelEncoder()
le_category = LabelEncoder()

df['publish_day_enc'] = le_day.fit_transform(df['publish_day'].astype(str))
df['channelTitle_enc'] = le_channel.fit_transform(df['channelTitle'].astype(str))
df['videoCategoryLabel_enc'] = le_category.fit_transform(df['videoCategoryLabel'].astype(str))

print('Encoding done.')
print('Days:', le_day.classes_)
print('Categories:', le_category.classes_)

---
## BLOC 4 — EDA

In [ ]:
# Distribution of views and engagement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(np.log1p(df['viewCount']), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of log(viewCount)')
axes[0].set_xlabel('log views')

er_vals = df['engagement_rate'].dropna()
axes[1].hist(er_vals, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribution of engagement_rate')
axes[1].set_xlabel('engagement rate')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = ['log_views', 'log_likes', 'log_comments', 'engagement_rate',
             'title_length', 'tags_count', 'is_short', 'is_weekend',
             'publish_hour', 'durationSec', 'has_emoji', 'high_view']
corr_cols = [c for c in corr_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 videos by viewCount
top_videos = df.nlargest(10, 'viewCount')[['channelTitle', 'videoTitle', 'viewCount', 'engagement_rate', 'is_short']]
print('=== Top 10 Videos by Views ===')
print(top_videos.to_string(index=False))

In [ ]:
# Shorts vs long-form comparison
format_comparison = df.groupby('is_short').agg(
    count=('videoId', 'count'),
    median_views=('viewCount', 'median'),
    median_engagement=('engagement_rate', 'median'),
    high_view_rate=('high_view', 'mean')
).rename(index={0: 'Long format', 1: 'Short (<= 60s}'})
print('=== Shorts vs Long Format ===')
print(format_comparison)

In [ ]:
# Performance by channel
channel_stats = df.groupby('channelTitle').agg(
    n_videos=('videoId', 'count'),
    median_views=('viewCount', 'median'),
    avg_engagement=('engagement_rate', 'mean'),
    pct_high_view=('high_view', 'mean')
).sort_values('median_views', ascending=False)

print('=== Channel Performance ===')
print(channel_stats.head(10).to_string())

In [ ]:
# Publication day vs high_view rate
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_perf = df.groupby('publish_day')['high_view'].mean().reindex(day_order)

fig, ax = plt.subplots(figsize=(9, 4))
day_perf.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('High-view rate by publication day')
ax.set_xlabel('Day')
ax.set_ylabel('Rate')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## BLOC 5 — XGBoost Modeling

In [ ]:
# Feature set for XGBoost — no raw string columns, no leakage from views
FEATURES = [
    'publish_hour', 'publish_day_enc', 'publish_month', 'is_weekend',
    'is_short', 'log_duration',
    'title_length', 'title_char_count', 'has_emoji', 'tags_count',
    'channelTitle_enc', 'videoCategoryLabel_enc',
    'licensedContent'
]
TARGET = 'high_view'

model_df = df[FEATURES + [TARGET, 'publishedAtSQL']].dropna()
print(f'Modeling dataset: {model_df.shape[0]} rows, {len(FEATURES)} features')

In [ ]:
# Temporal train/test split — simulates real-world prediction scenario
cutoff = model_df['publishedAtSQL'].quantile(0.80)
print(f'Train: before {cutoff.date()}')
print(f'Test:  on or after {cutoff.date()}')

train = model_df[model_df['publishedAtSQL'] < cutoff]
test = model_df[model_df['publishedAtSQL'] >= cutoff]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Train target balance: {y_train.mean():.3f}')
print(f'Test target balance:  {y_test.mean():.3f}')

In [ ]:
# XGBoost training
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

In [ ]:
# Evaluation
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Low view', 'High view']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low view', 'High view'],
            yticklabels=['Low view', 'High view'], ax=ax)
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = pd.Series(model.feature_importances_, index=FEATURES)
importance = importance.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('XGBoost Feature Importance')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

In [ ]:
# Save final cleaned dataset
df.to_csv('yt_morocco_benchmark_cleaned_v2.csv', index=False)
print(f'Final dataset saved: {df.shape[0]} rows, {df.shape[1]} columns')